In [1]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio 
import torchaudio.transforms as T

from datasets import load_dataset

import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd

In [32]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set


In [34]:
clean_dataset = Dataset(path, dev_split, None, 1)


Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


In [35]:
sampling_rate = 16000

In [36]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
model = model.cuda()


## Check performance on demo dataset

In [8]:
dataset2 = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
dataset2 = dataset2.sort("id")
sampling_rate = dataset2.features["audio"].sampling_rate

Reusing dataset librispeech_asr (/home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b)
Loading cached sorted indices for dataset at /home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b/cache-2f7c0cbee6ef3aa1.arrow


In [9]:
sampling_rate

16000

In [10]:
# results = []

model = model.cuda()
for ix in tqdm(range(5)):
    true = dataset2[ix]["text"]
    inputs = processor(dataset2[ix]["audio"]['array'], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
    print(f'guess: {transcription[0]}')
    print(f'truth: {true}')
    print()
    
#     results.append({'hyp': transcription[0],
#                     'truth': true})

  0%|          | 0/5 [00:00<?, ?it/s]

guess: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
truth: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

guess: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
truth: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER

guess: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
truth: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

guess: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LAYTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA
truth: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA

guess: LINNELL'S PIC

We can see this is a character model based on the handful of errors made. Overall performance looks good.

## Check on Single Word Corpora

no language model is getting used, so things may be messy 

In [50]:
clean_dataset.dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path'],
    num_rows: 200
})

In [51]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


In [52]:
updated_dataset = clean_dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


In [53]:
results = []

# model = model
for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    results.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [84]:
import pandas as pd

In [85]:
results = pd.DataFrame(results)

In [86]:
results['condition'] = 'clean'

In [87]:
results.head(5)

,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THE,THE,clean,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,clean,"[b'T', None]","[b'T', None]"
3,OF,OF,clean,"[b'AF', None]","[b'AF', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"


In [58]:
top_match_guesses = results[(results['hyp'] == results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

159 matched guesses; 79.5% correct


,hyp,truth,condition
0,THE,THE,clean
1,AND,AND,clean
3,OF,OF,clean
4,A,A,clean
5,THAT,THAT,clean
...,...,...,...
191,BETTER,BETTER,clean
194,BIG,BIG,clean
195,TWENTY,TWENTY,clean
196,EACH,EACH,clean


In [59]:
# For homophone matching 

dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

179 matched guesses; 89.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THE,THE,clean,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,clean,"[b'T', None]","[b'T', None]"
3,OF,OF,clean,"[b'AF', None]","[b'AF', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"
...,...,...,...,...,...
194,BIG,BIG,clean,"[b'PK', None]","[b'PK', None]"
195,TWENTY,TWENTY,clean,"[b'TNT', None]","[b'TNT', None]"
196,EACH,EACH,clean,"[b'AK', None]","[b'AK', None]"
197,SURE,SURE,clean,"[b'SR', None]","[b'SR', None]"


## On noise - 0 dBSNR

In [60]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

high_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = high_dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [61]:
results_0db = []

for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    results_0db.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [63]:
results_0db = pd.DataFrame(results_0db)
results_0db['condition'] = '0dB SNR'
top_match_guesses = results_0db[(results_0db['hyp'] == results_0db['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results_0db.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

119 matched guesses; 59.5% correct


,hyp,truth,condition
1,AND,AND,0dB SNR
5,THAT,THAT,0dB SNR
7,I,I,0dB SNR
9,IS,IS,0dB SNR
10,IT,IT,0dB SNR


In [64]:

results_0db['hyp_dmeta'] = results_0db['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results_0db['truth_dmeta'] = results_0db['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results_0db[(results_0db['hyp_dmeta'] == results_0db['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results_0db.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

141 matched guesses; 70.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THO,THE,0dB SNR,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,0dB SNR,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,0dB SNR,"[b'T', None]","[b'T', None]"
5,THAT,THAT,0dB SNR,"[b'0T', b'TT']","[b'0T', b'TT']"
7,I,I,0dB SNR,"[b'A', None]","[b'A', None]"


In [65]:
all_results = pd.concat([results, results_0db])

In [66]:
all_results

,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THE,THE,clean,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,clean,"[b'T', None]","[b'T', None]"
3,OF,OF,clean,"[b'AF', None]","[b'AF', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"
...,...,...,...,...,...
195,TWENTY,TWENTY,0dB SNR,"[b'TNT', None]","[b'TNT', None]"
196,E,EACH,0dB SNR,"[b'A', None]","[b'AK', None]"
197,SURE,SURE,0dB SNR,"[b'SR', None]","[b'SR', None]"
198,LET,LET,0dB SNR,"[b'LT', None]","[b'LT', None]"


## On noise at -5 dBSNR

In [67]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

low_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = low_dataset.dataset.map(load_audio)

Using custom data configuration default-5683877c4c4495f8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c3c5b179e5247f84.arrow


In [68]:
low_results = []

for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    low_results.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [79]:
low_results = pd.DataFrame(low_results)
low_results['condition'] = '-5dB SNR'
top_match_guesses = low_results[(low_results['hyp'] == low_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

54 matched guesses; 27.0% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
2,TO,TO,-5dB SNR,"[b'T', None]","[b'T', None]"
11,WAS,WAS,-5dB SNR,"[b'AS', b'FS']","[b'AS', b'FS']"
13,SO,SO,-5dB SNR,"[b'S', None]","[b'S', None]"
15,WE,WE,-5dB SNR,"[b'A', b'F']","[b'A', b'F']"
21,AS,AS,-5dB SNR,"[b'AS', None]","[b'AS', None]"


In [80]:

low_results['hyp_dmeta'] = low_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
low_results['truth_dmeta'] = low_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = low_results[(low_results['hyp_dmeta'] == low_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

74 matched guesses; 37.0% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
2,TO,TO,-5dB SNR,"[b'T', None]","[b'T', None]"
9,AS,IS,-5dB SNR,"[b'AS', None]","[b'AS', None]"
11,WAS,WAS,-5dB SNR,"[b'AS', b'FS']","[b'AS', b'FS']"
13,SO,SO,-5dB SNR,"[b'S', None]","[b'S', None]"
15,WE,WE,-5dB SNR,"[b'A', b'F']","[b'A', b'F']"


In [81]:
all_results = pd.concat([all_results, low_results])

In [82]:
all_results.condition.unique()

array(['clean', '0dB SNR', '-5dB SNR'], dtype=object)

In [83]:
# all_results.to_pickle('../single_word_results/wav2vec2_base_single_word.pdpkl')